In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from src.evaluate import compute_generation_metrics

sns.set_theme(style='whitegrid')

In [ ]:
baseline_rows = [json.loads(l) for l in open('../outputs/predictions/baseline_results.jsonl')]
qpp_rows      = [json.loads(l) for l in open('../outputs/predictions/adaptive_rag_results.jsonl')]

df_base = pd.DataFrame(baseline_rows)
df_qpp  = pd.DataFrame(qpp_rows)

print(f'Baseline:    {len(df_base)} queries')
print(f'QPP-guided:  {len(df_qpp)} queries')
print(df_qpp[['qpp_score', 'k_used', 'rouge_l', 'bert_f1', 'token_f1']].describe().round(4))

In [ ]:
df_compare = pd.DataFrame({
    'Method':    ['Baseline RAG', 'QPP-Guided RAG'],
    'ROUGE-L':   [df_base['rouge_l'].mean(),   df_qpp['rouge_l'].mean()],
    'BERTScore': [df_base['bert_f1'].mean(),   df_qpp['bert_f1'].mean()],
    'F1':        [df_base['token_f1'].mean(),  df_qpp['token_f1'].mean()],
}).set_index('Method')

df_compare.loc['Gain'] = df_compare.loc['QPP-Guided RAG'] - df_compare.loc['Baseline RAG']
df_compare.to_csv('../outputs/predictions/baseline_vs_qpp.csv')
print(df_compare.round(4))

In [ ]:
qpp_scores = df_qpp['qpp_score'].values.astype(float)
t1, t2 = np.percentile(qpp_scores, 33), np.percentile(qpp_scores, 66)

groups = {
    'Low-QPP':    qpp_scores < t1,
    'Medium-QPP': (qpp_scores >= t1) & (qpp_scores < t2),
    'High-QPP':   qpp_scores >= t2,
}

records = []
for label, mask in groups.items():
    b = df_base['rouge_l'].values[mask].mean()
    g = df_qpp['rouge_l'].values[mask].mean()
    records.append({
        'QPP Group':  label,
        'N':          int(mask.sum()),
        'Baseline':   round(b, 4),
        'QPP-Guided': round(g, 4),
        'Gain':       round(g - b, 4),
    })

df_groups = pd.DataFrame(records).set_index('QPP Group')
df_groups.to_csv('../outputs/predictions/gains_by_group.csv')
print(df_groups)

In [ ]:
# Retrieval depth distribution
k_counts = df_qpp['k_used'].value_counts().sort_index()
print('Retrieval depth distribution:')
print(k_counts)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Scatter: per-query ROUGE-L baseline vs QPP-guided
base_rouge = df_base['rouge_l'].values
qpp_rouge  = df_qpp['rouge_l'].values
scatter_colors = np.where(qpp_scores < t1, '#E53935',
                 np.where(qpp_scores < t2, '#FB8C00', '#43A047'))

axes[0].scatter(base_rouge, qpp_rouge, c=scatter_colors, alpha=0.5, s=15)
lim = [0.0, max(base_rouge.max(), qpp_rouge.max()) + 0.05]
axes[0].plot(lim, lim, 'k--', lw=1, alpha=0.5, label='No change')
axes[0].set_xlabel('Baseline ROUGE-L')
axes[0].set_ylabel('QPP-Guided ROUGE-L')
axes[0].set_title('Per-Query ROUGE-L Improvement')
patches = [
    mpatches.Patch(color='#E53935', label='Low-QPP  (k=50)'),
    mpatches.Patch(color='#FB8C00', label='Medium-QPP (k=30)'),
    mpatches.Patch(color='#43A047', label='High-QPP  (k=20)'),
]
axes[0].legend(handles=patches, fontsize=9)

# Bar: gains per group
group_labels   = df_groups.index.tolist()
baseline_vals  = df_groups['Baseline'].values
qpp_vals       = df_groups['QPP-Guided'].values
x = np.arange(len(group_labels))
w = 0.35
axes[1].bar(x - w/2, baseline_vals, w, label='Baseline RAG',  color='#90A4AE')
axes[1].bar(x + w/2, qpp_vals,      w, label='QPP-Guided RAG', color='#1E88E5')
for i, (b, g) in enumerate(zip(baseline_vals, qpp_vals)):
    axes[1].text(i + w/2, g + 0.003, f'+{g-b:.3f}', ha='center',
                 fontsize=9, color='#1E88E5', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(group_labels)
axes[1].set_ylabel('ROUGE-L')
axes[1].set_title('ROUGE-L Gain by QPP Group')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/adaptive_rag_gains.png', dpi=150, bbox_inches='tight')
plt.show()